In [19]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
DATA_PATH = '/gpfs/gibbs/pi/reilly/tabula_data'
PARAMS_PATH = '../model_fitting'

In [3]:
class DataSet(dict):
    def __init__(self, path):
        self.filepath = path
        self.parquet = pq.ParquetFile(self.filepath)
    
    def __getitem__(self, key):
        try:
            return self.parquet.read([key]).to_pandas()[key]
        except:
            raise KeyError

    def __reduce__(self):
        #return self.parquet.read().to_pandas().__reduce__()
        return (self.__class__, (self.filepath, ))

# Combo dataset, grouped by CRE/Cell Type

Convert to consistent formatting and naming 

Each entry is an MPRA barcode CRE measured in a given cell.

In [6]:
counts = pd.read_table('%s/shendure/GSE217686_assigned_oBC_CRE_mBC_joined_counts_sc_rep_mEB_series.txt' % DATA_PATH)
counts.columns = ['cell_bc', 'rep_id', 'transfection_bc', 'mpra_bc', 'cre_class', 'cre_id'
                    ,'reads_transfection_bc', 'umis_transfection_bc', 
                    'reads_mpra_bc', 'umis_mpra_bc']


In [8]:
cell_type_mapping = pd.read_table('%s/shendure/meb_cbc_to_cell_type_mapping.txt' % DATA_PATH)

In [9]:
cell_type_mapping.loc[counts['cell_bc']]

,orig.ident,nCount_RNA,nFeature_RNA,percent_mt,seurat_clusters,annotation
A1_GTTACCCAGTTGAAGT-1,repA1,3913,1664,4.702274,3,Surface Ectoderm
A1_GTTACCCAGTTGAAGT-1,repA1,3913,1664,4.702274,3,Surface Ectoderm
A1_GTTACCCAGTTGAAGT-1,repA1,3913,1664,4.702274,3,Surface Ectoderm
A1_GTTACCCAGTTGAAGT-1,repA1,3913,1664,4.702274,3,Surface Ectoderm
A1_GTTACCCAGTTGAAGT-1,repA1,3913,1664,4.702274,3,Surface Ectoderm
...,...,...,...,...,...,...
2B2_TTCTGTAAGGCTCTCG-1,rep2B2,522,384,5.747126,7,Neuroectoderm (rostral)
2B2_TTCTGTAAGGCTCTCG-1,rep2B2,522,384,5.747126,7,Neuroectoderm (rostral)
2B2_TGGAACTCAATCCTAG-1,rep2B2,968,619,1.136364,2,Mesoderm
2B2_TGGAACTCAATCCTAG-1,rep2B2,968,619,1.136364,2,Mesoderm


In [10]:
counts['cell_type'] = np.array(cell_type_mapping.loc[counts['cell_bc']]['annotation'])
counts['gex_umi'] = np.array(cell_type_mapping.loc[counts['cell_bc']]['nCount_RNA'])
# Compute mean GEx UMIs across all cells
mean_gex_umi = counts['gex_umi'].mean()

# Perform the normalization as described
counts['normalized_umis_mpra_bc'] = (
    counts['umis_mpra_bc'] / counts['gex_umi'] * mean_gex_umi
)

In [11]:
counts = counts.replace({'Epiblast/primitive streak' : 'EpiblastPrimitiveStreak', 
                            'Ex. Endoderm (parietal)' : 'ExEndodermParietal',
                            'Ex. Endoderm (visceral)': 'ExEndodermVisceral',
                            'Neuroectoderm (brain)' : 'NeuroectodermBrain',
                            'Neuroectoderm (rostral)' : 'NeuroectodermRostral',
                            'Surface Ectoderm' : 'SurfaceEctoderm'})

In [12]:
counts.to_csv('%s/shendure/shendure_counts_not_grouped.txt' % DATA_PATH, sep='\t', index=False)

In [13]:
counts_groupby_cre = counts.groupby(by=['cell_bc','rep_id','cre_class','cre_id','cell_type'], as_index=False).agg(lambda x: x.sum() if np.issubdtype(x.dtype, np.number) else ', '.join(x))
counts_groupby_cre 
counts_groupby_cre.to_csv('%s/shendure/shendure_counts_grouped.txt' % DATA_PATH, sep='\t', index=False)

In [ ]:
counts_groupby_cre

,cell_bc,rep_id,cre_class,cre_id,cell_type,transfection_bc,mpra_bc,reads_transfection_bc,umis_transfection_bc,reads_mpra_bc,umis_mpra_bc,gex_umi,normalized_umis_mpra_bc
0,2B1_AAACCCACAAGGTTGG-1,2B1,devCRE,Btg1_chr10_9572,NeuroectodermBrain,CATCGCTGAGTAAACG,GCGTACTCACCAGGT,73,67,0,0,2376,0.000000
1,2B1_AAACCCACAAGGTTGG-1,2B1,devCRE,Gata4_chr14_5710,NeuroectodermBrain,GAGGATGAGTTGGAAT,CAATCGCACCCCCGA,167,159,0,0,2376,0.000000
2,2B1_AAACCCACAAGGTTGG-1,2B1,devCRE,Klf4_chr4_3952,NeuroectodermBrain,CGTGAATTAATTCTAT,AACCCGGTAAATGTA,99,89,0,0,2376,0.000000
3,2B1_AAACCCACAAGGTTGG-1,2B1,devCRE,Lama1_chr17_7793,NeuroectodermBrain,AGTAAGTCAGCTCTTT,CGTGACCTCTTCATT,159,147,0,0,2376,0.000000
4,2B1_AAACCCACAAGGTTGG-1,2B1,devCRE,Sox17_chr1_67,NeuroectodermBrain,GACAATAAAATTCCAT,ACAGTCACAAATTTA,59,58,0,0,2376,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
778243,B2_TTTGTTGGTGGACCAA-1,B2,devCRE,Sparc_chr11_7207,ExEndodermVisceral,AATAATCACTCAAATT,TACCAACTGAGACAT,130,104,0,0,1438,0.000000
778244,B2_TTTGTTGGTGGACCAA-1,B2,devCRE,Tgfbi_chr13_5735,ExEndodermVisceral,AGTCCATGGAGGGAGG,GTTTACCACATTACT,108,98,0,0,1438,0.000000
778245,B2_TTTGTTGGTGGACCAA-1,B2,devCRE,Tgfbi_chr13_5741,ExEndodermVisceral,CTCAAGTTAGTAAGGG,CAGGGAACTGCCACC,75,68,19,1,1438,2.082943
778246,B2_TTTGTTGGTGGACCAA-1,B2,promoters,noP,ExEndodermVisceral,"ACTTCTCGCCAAGGAA, GTTTCTTCGTCTGCCC","TCCCGCTGACACTTA, ATGTGGGTCGTCTAT",200,166,0,0,2876,0.000000


: 

In [15]:
table = pa.Table.from_pandas(counts)
pq.write_table(table, '%s/shendure/shendure_mpra_counts.parq' % DATA_PATH)
counts_parq = DataSet('%s/shendure/shendure_mpra_counts.parq' % DATA_PATH)


table = pa.Table.from_pandas(counts_groupby_cre)
pq.write_table(table, '%s/shendure/shendure_mpra_counts_grouped.parq' % DATA_PATH)
counts_parq_grouped = DataSet('%s/shendure/shendure_mpra_counts_grouped.parq' % DATA_PATH)

In [9]:
counts_groupby_cre

,cell_bc,rep_id,cre_class,cre_id,cell_type,transfection_bc,mpra_bc,reads_transfection_bc,umis_transfection_bc,reads_mpra_bc,umis_mpra_bc
0,2B1_AAACCCACAAGGTTGG-1,2B1,devCRE,Btg1_chr10_9572,NeuroectodermBrain,CATCGCTGAGTAAACG,GCGTACTCACCAGGT,73,67,0,0
1,2B1_AAACCCACAAGGTTGG-1,2B1,devCRE,Gata4_chr14_5710,NeuroectodermBrain,GAGGATGAGTTGGAAT,CAATCGCACCCCCGA,167,159,0,0
2,2B1_AAACCCACAAGGTTGG-1,2B1,devCRE,Klf4_chr4_3952,NeuroectodermBrain,CGTGAATTAATTCTAT,AACCCGGTAAATGTA,99,89,0,0
3,2B1_AAACCCACAAGGTTGG-1,2B1,devCRE,Lama1_chr17_7793,NeuroectodermBrain,AGTAAGTCAGCTCTTT,CGTGACCTCTTCATT,159,147,0,0
4,2B1_AAACCCACAAGGTTGG-1,2B1,devCRE,Sox17_chr1_67,NeuroectodermBrain,GACAATAAAATTCCAT,ACAGTCACAAATTTA,59,58,0,0
...,...,...,...,...,...,...,...,...,...,...,...
778243,B2_TTTGTTGGTGGACCAA-1,B2,devCRE,Sparc_chr11_7207,ExEndodermVisceral,AATAATCACTCAAATT,TACCAACTGAGACAT,130,104,0,0
778244,B2_TTTGTTGGTGGACCAA-1,B2,devCRE,Tgfbi_chr13_5735,ExEndodermVisceral,AGTCCATGGAGGGAGG,GTTTACCACATTACT,108,98,0,0
778245,B2_TTTGTTGGTGGACCAA-1,B2,devCRE,Tgfbi_chr13_5741,ExEndodermVisceral,CTCAAGTTAGTAAGGG,CAGGGAACTGCCACC,75,68,19,1
778246,B2_TTTGTTGGTGGACCAA-1,B2,promoters,noP,ExEndodermVisceral,"ACTTCTCGCCAAGGAA, GTTTCTTCGTCTGCCC","TCCCGCTGACACTTA, ATGTGGGTCGTCTAT",200,166,0,0


# Cell type individual groupings

In [11]:
counts_groupby_cell_agg = counts_groupby_cre.groupby(['cell_type']).agg(
    Sum=('umis_mpra_bc', 'sum'), Size=('umis_mpra_bc','size'), Mean=('umis_mpra_bc', 'mean')
)
counts_groupby_cell_agg

,Sum,Size,Mean
cell_type,,,
Cardiomyocytes,69992,12037,5.814738
EpiblastPrimitiveStreak,944637,66949,14.109800
ExEndodermParietal,232424,77898,2.983697
ExEndodermVisceral,353833,51713,6.842245
Haematoendothelial,169728,21749,7.803945
Mesoderm,452921,128596,3.522046
NeuroectodermBrain,665793,135976,4.896401
NeuroectodermRostral,52329,30630,1.708423
Pluripotent,1800660,165773,10.862203


In [12]:
cell_types = list(counts_groupby_cell_agg.index)
len(cell_types)

10

In [13]:
for i in cell_types:
    print(i)
    counts = counts_groupby_cre[counts_groupby_cre.cell_type == i]
    table = pa.Table.from_pandas(counts)
    pq.write_table(table, '%s/shendure/cell_type/shendure_mpra_counts_%s.parq' % (DATA_PATH, i))

Cardiomyocytes
EpiblastPrimitiveStreak
ExEndodermParietal
ExEndodermVisceral
Haematoendothelial
Mesoderm
NeuroectodermBrain
NeuroectodermRostral
Pluripotent
SurfaceEctoderm
